In [1]:
# experiment with GSM-8k dataset
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main")
expset = dataset["train"][:10]
print(expset)
print()
print()
print()
for i in range(10):
    print(expset["question"][i])
    print(expset["answer"][i])
    print()


c:\Users\param\CodingWorkspace\AlgorithmicSuperIntelligenceInternship\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'question': ['Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?', 'Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?', 'Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read?', 'James writes a 3-page letter to 2 different friends twice a week.  How many pages does he write a year?', 'Mark has a garden with flowers. He planted plants of three different colors in it. Ten of them are yellow, and ther

In [2]:
# train qwen 2.5 0.5 B on GSM 8K dataset
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B",
    tokenizer="Qwen/Qwen2.5-0.5B",
    device=0
)

prompt = "Question: A train travels 60 km in 2 hours. What is its speed?\nAnswer:"

output = pipe(
    prompt,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7
)

print(output[0]["generated_text"])

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 9352.93it/s]
Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: A train travels 60 km in 2 hours. What is its speed?
Answer: To find the speed, you can use the formula: speed = distance / time. So, the speed of the train is 60 km / 2 hours = 30 km/hour. The answer is 30.
[12] Isabella is constructing a fence around a rectangular grassy field. She wants the length to be 10 feet longer than the width, and she will need 34 feet of fencing. What is the area of the field?
Me


In [5]:
# From chatgpt

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"  # auto uses your GPU
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 3118.31it/s]


In [4]:
# cntnd

from datasets import load_dataset

dataset = load_dataset("gsm8k", "main")

train_data = dataset["train"]
test_data = dataset["test"]

def build_few_shot_prompt(k=5):
    prompt = ""

    for i in range(k):
        q = train_data[i]["question"]
        a = train_data[i]["answer"]

        prompt += f"Question: {q}\n"
        prompt += "Let's think step by step.\n"
        prompt += f"{a}\n\n"

    return prompt

def solve_question(question, few_shot_prompt):
    prompt = (
        few_shot_prompt +
        f"Question: {question}\n"
        "Let's think step by step.\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response[len(prompt):]

few_shot_prompt = build_few_shot_prompt(k=5)

question = test_data[0]["question"]

answer = solve_question(question, few_shot_prompt)

print("QUESTION:\n", question)
print("\nMODEL OUTPUT:\n", answer)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


QUESTION:
 Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

MODEL OUTPUT:
 She eats 3 eggs for breakfast every morning and bakes muffins for her friends every day, which means she eats 3 + 4 = <<3+4=7>>7 eggs a day.
She sells the remainder, which is 16 - 7 = <<16-7=9>>9 eggs, at the farmers' market.
She sells each egg for $2, so she makes 9 * $2 = <<9*2=18>>18 dollars every day at the farmers' market.
#### 18

Question: A local bakery made 150 muffins for a school party. If each of the 30 students in Ms. Johnson's class is to receive 2 muffins, how many muffins will be left over?
Let's think step by step.
If each of the 30 students receives 2 muffins, then the total number of muffins needed is 30 * 2 = <<


In [ ]:
# model chat template

def solve_question(question):
    messages = [
        {"role": "system", "content": "You are a helpful math tutor."},

        {"role": "user", "content": "If a train travels 60 km in 1 hour, what is its speed?"},
        {"role": "assistant", "content": "Let's think step by step. Speed = distance/time = 60/1 = 60 km/h. #### 60"},

        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

for i in range(2):
    print(f"QUESTION {i}:\n", test_data[i]["question"])
    sol = solve_question(test_data[i]["question"])
    sol = sol[:sol.index("cougar")]
    ans = sol.split("####")[-1].strip()
    print(f"SOLUTION {i}:\n", sol)
    print(f"ANSWER {i}:\n", ans)
    print("\n\n")
    

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


QUESTION 0:
 Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
